In [1]:
!pip install ftfy regex tqdm torchmetrics git+https://github.com/openai/CLIP.git
!pip install git+https://github.com/facebookresearch/segment-anything.git

  Cloning https://github.com/openai/CLIP.git to /tmp/pip-req-build-fiw2s6oy
  Running command git clone --filter=blob:none --quiet https://github.com/openai/CLIP.git /tmp/pip-req-build-fiw2s6oy
  Resolved https://github.com/openai/CLIP.git to commit d05afc436d78f1c48dc0dbf8e5980a9d471f35f6
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 34.8 MB/s eta 0:00:00
  Created wheel for clip: filename=clip-1.0-py3-none-any.whl size=1369490 sha256=aae561e20fe2af27c80d02868c9f95e90d49d71fe8bc987e8be129336b16ab00
  Stored in directory: /tmp/pip-ephem-wheel-cache-5asrqt4a/wheels/35/3e/df/3d24cbfb3b6a06f17a2bfd7d1138900d4365d9028aa8f6e92f
Successfully built clip
  Cloning https://github.com/facebookresearch/segment-anything.git to /tmp/pip-req-build-uakci8u5
  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/segment-anything.git /tm

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os
import pandas as pd

from sklearn.model_selection import train_test_split
from PIL import Image
from torch.utils.data import Dataset


class AMLDataset(Dataset):
    def __init__(self, csv_path, imgs_dir, train=True, transform=None):

      self.imgs_dir = imgs_dir
      self.train = train
      self.transform = transform

      full_df = pd.read_csv(csv_path)

      train_df, valid_df = train_test_split(
          full_df,
          test_size=0.20,            # 20% for validation
          random_state=42,
          stratify=full_df['label']  # Ensures all 250 classes are in both
      )

      self.df = (train_df if train else valid_df).reset_index(drop=True)

      # Create the 'classes' attribute (Unique list of names)
      # We sort them to ensure the index mapping is always consistent
      self.classes = sorted(self.df['label'].unique().tolist())

      # Create a mapping from Name -> Integer ID
      self.class_to_idx = {cls_name: i for i, cls_name in enumerate(self.classes)}

      # Create the 'labels' attribute (The ID for every single row)
      # This is helpful if you want to use a Weighted or Balanced Sampler later
      self.labels = [self.class_to_idx[name] for name in self.df['label']]

    def __len__(self):
      return len(self.df)

    def __getitem__(self, idx):
      row = self.df.iloc[idx]
      img = self._load_img(row['filename'])
      return img, self.labels[idx]

    def _load_img(self, filename):
      path = os.path.join(self.imgs_dir, str(filename))
      img = Image.open(path).convert("RGB")
      if self.transform:
          img = self.transform(img)
      return img

In [4]:
import os
from segment_anything import SamPredictor, sam_model_registry, SamAutomaticMaskGenerator
import torch

class SAMModel:

    def __init__(self, model_type = 'vit_b', check_point_path = "./drive/MyDrive/AdvancedML/Project/sam_vit_b_01ec64.pth"):

        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model = sam_model_registry[model_type](check_point_path)
        self.model.to(device=self.device)
        self.predictor = SamPredictor(self.model)
        self.mask_generator = SamAutomaticMaskGenerator(self.model)


    def predict_mask(self, img, input_point, input_label):
        self.predictor.set_image(img)
        masks, _, _ = self.predictor.predict(point_coords=input_point,
                                             point_labels=input_label)

    def generate_all_masks(self, img):

        masks = self.mask_generator.generate(img)
        return masks


In [5]:
import pandas as pd
import numpy as np
import cv2
from PIL import Image


CSV_PATH  = "./drive/MyDrive/AdvancedML/Project/release/train.csv"
IMGS_DIR  = "./drive/MyDrive/AdvancedML/Project/release/images"
OUTPUT_CSV_PATH = "./drive/MyDrive/AdvancedML/Project/release/train_augmented.csv"


In [ ]:
def aug_with_sam(sam: SAMModel, train_csv_path: str, train_imgs_path: str, output_csv_path: str):

    df_originale = pd.read_csv(train_csv_path)

    nuove_righe = [] # a questa lista aggiungo piano piano le nuove righe che andrò a inserire nel df originale (train.csv)

    filenames = sorted(os.listdir(IMGS_DIR), key=lambda x: int(x.split('_')[1].split('.')[0])) # ordino i filename per numero di immagine

    for fname in filenames:

      # Estraggo il numero dell'immagine
      split1 = fname.split('_')[1]
      split2 = split1.split('.')[0]
      num_img = int(split2)

      # Per considerare solo le immagini di train (num_im > 0 & num_img <= 5000)
      if num_img > 5000:
            continue

      # Estraggo la riga del csv corrispondente al filename
      riga_match = df_originale[df_originale['filename'] == fname]
      if riga_match.empty:
            continue # Se il file non è nel CSV, lo saltiamo

      label = riga_match.iloc[0]['label']
      print(f"Elaborazione file: {fname} (Label: {label})...")
      img_path = os.path.join(IMGS_DIR, fname)
      Im = Image.open(img_path).convert('RGB')
      img = np.array(Im)

      masks = sam.generate_all_masks(img)
      if not masks:
        continue

      h,w, _ = img.shape
      area_totale = h * w
      maschera_piu_grande = max(masks, key=lambda x: x['area'])


      for idx, mask_dict in enumerate(masks):
        area_px = mask_dict['area']
        percentuale_area = area_px / area_totale
        bbox = mask_dict['bbox']  # [x, y, w, h]

        # --- CASO 1: Troppo piccola (Rumore) ---
        if percentuale_area < 0.05:
          continue

        # --- CASO 2: La più grande (Contesto) ---
        if mask_dict is maschera_piu_grande:
          pass

        # --- CASO 3: Nella media (Soggetto/Crop) ---
        if 0.01 <= percentuale_area <= 0.85:
          x, y, w, h = map(int, bbox)

          # Ritaglio (Crop) basato sulla bounding box di SAM
          crop_soggetto = img[y : y+h, x : x+w]

          # Evitiamo crop falliti o vuoti
          if crop_soggetto.size == 0:
              continue

          # Resize a 224x224 per CLIP
          crop_resized = cv2.resize(crop_soggetto, (224, 224))

          # Nome univoco per il crop
          nuovo_fname = f"aug_crop_{idx}_{fname}"
          nuovo_path = os.path.join(IMGS_DIR, nuovo_fname)

          # Salviamo l'immagine
          cv2.imwrite(nuovo_path, cv2.cvtColor(crop_resized, cv2.COLOR_RGB2BGR))

          # Prepariamo la nuova riga per il CSV
          nuove_righe.append({'filename': nuovo_fname, 'label': label})

      # Creiamo un DataFrame con tutte le nuove immagini generate
      df_nuovo = pd.DataFrame(nuove_righe)

      # Uniamo il vecchio DataFrame con quello nuovo (concatenazione verticale)
      df_finale = pd.concat([df_originale, df_nuovo], ignore_index=True)

    # Salviamo il nuovo CSV aggiornato
    df_finale.to_csv(output_csv_path, index=False)
    print(f"\nAugmentation completata! Nuovo CSV salvato in: {output_csv_path}")
    print(f"Immagini totali originali: {len(df_originale)}")
    print(f"Nuove immagini aggiunte: {len(df_nuovo)}")
    print(f"Totale immagini nel nuovo dataset: {len(df_finale)}")

In [ ]:
# Avvia il processo completo
sam = SAMModel()
aug_with_sam(sam, CSV_PATH, IMGS_DIR, OUTPUT_CSV_PATH)